# Assignment 3 — Clasificadores Probabilísticos: NB, LDA y QDA
## Unidad 2

**versión: 2025-1  |  modificado: 2026-04-06**

> 📝 **Modalidad:** Trabajo autónomo — practica a tu ritmo.

---

<div style="background-color:#F8F9FA; border:2px solid #AEB6BF;
     padding:12px 18px; border-radius:8px; margin:10px 0;">
<strong>🎓 Modo de uso:</strong> Este notebook es compartido por dos cursos.<br><br>
<span style="color:#2E86C1; font-weight:bold;">🔵 Pregrado</span>
— Trabaja el contenido general y los bloques azules.<br><br>
<span style="color:#B7950B; font-weight:bold;">🟡 Doctorado</span>
— Trabaja el contenido general y ambos bloques.
</div>

---

## Rúbrica Orientativa

> ⚠️ Esta rúbrica es orientativa. La nota final se asigna en la prueba escrita, no en este notebook.

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado — Competencias a demostrar</span><br><br>

- Aplicar el teorema de Bayes para clasificar con distribuciones gaussianas.
- Entrenar y comparar GNB, LDA y QDA con validación cruzada.
- Identificar cuándo el supuesto de independencia de NB es problemático.
- Interpretar fronteras de decisión lineales y cuadráticas visualmente.
- Elegir y justificar el modelo adecuado según las características del dataset.

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ Competencias adicionales en el bloque 🟡 Doctorado.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado — Competencias adicionales</span><br><br>

- Todo lo anterior más:
- Derivar la función discriminante cuadrática y verificarla implementando QDA desde cero.
- Analizar el efecto de la correlación entre features en el desempeño de NB.
- Comparar la covarianza compartida (LDA) vs por clase (QDA) usando criterios estadísticos.
- Diseñar y ejecutar un experimento de ablación sistemático.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de estas competencias está en el bloque 🔵 Pregrado.</span>
</div>

---
## Contexto del Problema

El **dataset Penguin** (Palmer Penguins) contiene mediciones morfológicas de 344 pingüinos de tres especies: Adelie, Chinstrap y Gentoo, recogidas en el archipiélago Palmer (Antártida). Las features incluyen longitud y profundidad del pico, longitud de la aleta y masa corporal.

El objetivo es construir un clasificador de especie que sea **interpretable** (el biólogo de campo quiere entender qué feature discrimina mejor) y **robusto** (funciona bien con pocos datos, ya que la recolección en campo es costosa).

Este problema es ideal para los tres modelos de la clase porque las especies tienen **distribuciones morfológicas distintas** y algunas features tienen **alta correlación** entre sí.

**Restricciones:**
- No usar modelos de ensemble ni redes neuronales.
- Cualquier imputación de valores faltantes debe hacerse en el pipeline, no antes del split.
- No mirar el test set hasta la sección de evaluación final.

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import __version__ as sklearn_version
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

# dataset: penguins_lter.csv  |  md5: generado desde seaborn (built-in)
# Fuente: Gorman et al. (2014). doi:10.1371/journal.pone.0090081
try:
    df_raw = sns.load_dataset('penguins')
    print(f'✅ Dataset penguins cargado — shape: {df_raw.shape}')
except Exception:
    # Fallback sintético
    from sklearn.datasets import make_classification
    X_s, y_s = make_classification(n_samples=344, n_features=4, n_informative=3,
                                   n_classes=3, n_clusters_per_class=1,
                                   random_state=RANDOM_STATE)
    df_raw = pd.DataFrame(X_s, columns=['bill_length_mm','bill_depth_mm',
                                         'flipper_length_mm','body_mass_g'])
    df_raw['species'] = pd.Categorical.from_codes(y_s, ['Adelie','Chinstrap','Gentoo'])
    print('⚠️  Usando dataset sintético (seaborn no disponible)')

# Preprocesamiento mínimo
FEATURES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
TARGET   = 'species'
df = df_raw[FEATURES + [TARGET]].copy()

X = df[FEATURES]
y = df[TARGET].astype('category').cat.codes   # Adelie=0, Chinstrap=1, Gentoo=2
SPECIES_NAMES = df[TARGET].astype('category').cat.categories.tolist()

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'✅ Setup completo')
print(f'   numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn_version}')
print(f'   Train: {X_train.shape} · Test: {X_test.shape}')
print(f'   Especies: {SPECIES_NAMES}')
print(f'   Valores faltantes en train: {X_train.isnull().sum().sum()}')

---
## Sección 1 — EDA y Exploración

In [ ]:
# ━━━ SECCIÓN 1: EDA ━━━

print('=== Estadísticas por especie (training set) ===')
print(X_train.assign(species=y_train.map(dict(enumerate(SPECIES_NAMES))))
             .groupby('species').describe().round(2))

# Pairplot para visualizar separabilidad y correlaciones
df_train_plot = X_train.copy()
df_train_plot['species'] = y_train.map(dict(enumerate(SPECIES_NAMES)))

g = sns.pairplot(df_train_plot.dropna(), hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.5, 's': 25})
g.fig.suptitle('Pairplot — Penguins Dataset (training set)', y=1.01,
               fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlaciones
print('\n=== Matriz de correlación de features (train) ===')
print(X_train.corr().round(3))

---
## Sección 2 — Modelos Base con Pipeline

In [ ]:
# ━━━ SECCIÓN 2: PIPELINES ━━━
# Nota: incluimos SimpleImputer para manejar los NaN del dataset Penguins

# Pipeline base para GNB (no necesita scaler, pero sí imputer)
pipe_gnb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', GaussianNB())
])

# LDA y QDA tampoco requieren scaler (son invariantes a escala lineal),
# pero incluir imputer es necesario para este dataset
pipe_lda = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', LinearDiscriminantAnalysis())
])

pipe_qda = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', QuadraticDiscriminantAnalysis(reg_param=0.01))  # pequeña regularización
])

pipelines = {'Gaussian NB': pipe_gnb, 'LDA': pipe_lda, 'QDA': pipe_qda}

# Entrenamiento y evaluación en test set
print('=== Evaluación en Test Set ===')
for nombre, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    print(f'{nombre:15s} → Accuracy: {acc:.3f}  F1-macro: {f1:.3f}')

print()
print('--- Classification report del mejor modelo ---')
# Mostrar el reporte del modelo con mayor F1 (evaluar cuál es en tu ejecución)
y_pred_lda = pipe_lda.predict(X_test)
print(classification_report(y_test, y_pred_lda, target_names=SPECIES_NAMES))

In [ ]:
# ─── TESTS DE SANIDAD — SECCIÓN 2 — NO MODIFICAR ───
print('=== Tests de Sanidad — Sección 2 ===')
try:
    for nombre, pipe in pipelines.items():
        y_p = pipe.predict(X_test)
        assert len(y_p) == len(y_test)
        assert set(y_p) <= {0, 1, 2}
        acc = accuracy_score(y_test, y_p)
        assert 0.0 <= acc <= 1.0
    print('✅ Los 3 pipelines predicen correctamente (tamaño, clases, rango accuracy)')
except AssertionError as e:
    print(f'❌ {e}')

try:
    for nombre, pipe in pipelines.items():
        imputer = pipe.named_steps.get('imputer')
        assert imputer is not None, f'{nombre}: falta el imputer en el pipeline'
        assert hasattr(imputer, 'statistics_'), f'{nombre}: el imputer no fue fiteado'
    print('✅ Imputer presente y fiteado en todos los pipelines (no data leakage en NaN)')
except AssertionError as e:
    print(f'❌ {e}')

---
## Sección 3 — Comparación con Validación Cruzada

In [ ]:
# ━━━ SECCIÓN 3: CV COMPARATIVA ━━━

CV_STR  = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
METRICAS = ['accuracy', 'f1_macro']

resultados_cv = {}
for nombre, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=CV_STR, scoring=METRICAS)
    resultados_cv[nombre] = {
        m: (scores[f'test_{m}'].mean(), scores[f'test_{m}'].std())
        for m in METRICAS
    }

print('=== Comparación 10-Fold CV — Penguins ===')
print(f'{"Modelo":15s} {"Accuracy":>14s} {"F1-macro":>14s}')
print('-' * 46)
for nombre, res in resultados_cv.items():
    acc_m, acc_s = res['accuracy']
    f1_m,  f1_s  = res['f1_macro']
    print(f'{nombre:15s} {acc_m:.3f} ± {acc_s:.3f}  {f1_m:.3f} ± {f1_s:.3f}')

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_idx, metrica in enumerate(METRICAS):
    ax = axes[ax_idx]
    all_scores = [
        cross_val_score(pipe, X, y, cv=CV_STR, scoring=metrica)
        for pipe in pipelines.values()
    ]
    ax.boxplot(all_scores, labels=list(pipelines.keys()), patch_artist=True)
    ax.set_title(f'{metrica.replace("_"," ").title()} — 10-Fold CV',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel(metrica.replace('_', ' ').title(), fontsize=11)

plt.suptitle('Comparación de modelos — Penguins', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── TESTS DE SANIDAD — SECCIÓN 3 — NO MODIFICAR ───
print('=== Tests de Sanidad — Sección 3 ===')
try:
    assert len(resultados_cv) == 3, '3 modelos requeridos en la CV'
    for nombre, res in resultados_cv.items():
        acc_m, acc_s = res['accuracy']
        assert 0 <= acc_m <= 1, f'{nombre}: accuracy media fuera de rango'
        assert 0 <= acc_s <= 0.5, f'{nombre}: std accuracy fuera de rango'
    print('✅ Resultados CV en rangos válidos')
except (AssertionError, KeyError) as e:
    print(f'❌ {e}')

print('\n✅ Sanity checks OK — tu implementación parece correcta.')
print('   Recuerda revisar también las preguntas escritas de reflexión.')

---
## Sección 4 — Análisis Diferenciado

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

1. Mirando el pairplot, ¿las tres especies tienen formas (distribuciones) similares o distintas? ¿Qué implica eso para la elección entre LDA y QDA?

2. La correlación entre `flipper_length_mm` y `body_mass_g` es alta (~0.87). ¿Esperarías que esto perjudique a NB respecto a LDA? ¿El resultado de tu CV confirma esa hipótesis?

3. Un biólogo de campo quiere un modelo que pueda explicar en términos de mediciones morfológicas. ¿Qué modelo le recomendarías y cómo le explicarías la regla de clasificación?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ver el análisis formal? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<!-- RESPUESTA MODELO:
P1: Gentoo tiene distribución muy diferente (mayor flipper, mayor masa). Adelie y Chinstrap son más similares pero con distintas profundidades de pico. Covarianzas distintas → QDA tiene ventaja teórica, pero con n=344 puede sobreajustar.
P2: Alta correlación viola el supuesto de independencia de NB. Si LDA > NB en CV, confirma que la correlación es relevante. La diferencia suele ser pequeña porque NB es robusto a correlaciones moderadas en la práctica.
P3: LDA es el más interpretable: da un vector de coeficientes directamente. "Si la combinación lineal 0.4*flipper - 0.2*bill_depth + ... > umbral, es Gentoo". NB también es interpretable (umbral por feature), QDA es el menos explicable.
-->

**Respuesta 1:** *(escribe aquí)*

**Respuesta 2:** *(escribe aquí)*

**Respuesta 3:** *(escribe aquí)*

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

4. Implementa QDA **desde cero** (sin sklearn) para el subproblema binario Adelie vs Gentoo usando solo `flipper_length_mm` y `body_mass_g`. Verifica que coincide con sklearn QDA. Grafica las gaussianas de cada clase y la frontera cuadrática.

5. Diseña y ejecuta un **experimento de ablación**: varía el tamaño del training set (10%, 25%, 50%, 75%, 100%) y mide cómo cambia el F1-macro de cada modelo. ¿En qué tamaño QDA empieza a superar a LDA? ¿Y NB a LDA?

6. LDA produce una proyección en $K-1=2$ dimensiones. Visualiza los datos proyectados en el espacio LDA coloreados por especie, y muestra la varianza explicada por cada eje. Interpreta qué feature original contribuye más a cada eje discriminante usando `lda.coef_`.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de estas derivaciones está en el bloque 🔵 Pregrado.</span>
</div>

<!-- RESPUESTA MODELO:
P4: Calcular mu_k, Sigma_k para k in {Adelie, Gentoo}. log_posterior_k = log(pi_k) - 0.5*log|Sigma_k| - 0.5*(x-mu_k).T @ inv(Sigma_k) @ (x-mu_k). Predicción: argmax_k. Frontera: curva implícita donde los dos log-posteriores son iguales.
P5: Curva de aprendizaje comparativa. Hipótesis: QDA necesita más datos para estimar 2 matrices 2x2 = 6 params por clase. LDA solo estima 1 matriz 4x4 = 10 params. NB solo 2*4=8 params. Con pocos datos, NB y LDA dominan; con más datos, QDA puede ganar.
P6: lda.coef_ da los vectores de proyección. Feature con mayor |coef| en LD1 es la más discriminante. En Penguins, flipper_length y body_mass suelen dominar LD1 (separa Gentoo de las demás). bill_depth domina LD2 (separa Adelie de Chinstrap).
-->

In [ ]:
# [PhD] Espacio para los análisis de doctorado

# Pregunta 4: QDA desde cero — subproblema Adelie vs Gentoo
# TODO [PhD]: implementa aquí


In [ ]:
# [PhD] Pregunta 5: Experimento de ablación por tamaño de train
# TODO [PhD]: implementa aquí


In [ ]:
# [PhD] Pregunta 6: LDA como reducción de dimensionalidad + interpretación
# TODO [PhD]: implementa aquí


**Respuesta 4 [PhD]:** *(escribe aquí)*

**Respuesta 5 [PhD]:** *(escribe aquí)*

**Respuesta 6 [PhD]:** *(escribe aquí)*

---
## ✅ Sección 5 — Autoevaluación

### Pregrado
- [ ] EDA completado con pairplot e interpretación de correlaciones
- [ ] Los 3 pipelines entrenados y evaluados en test set
- [ ] CV comparativa con 10 folds
- [ ] Tests de sanidad: todos ✅ PASS
- [ ] Preguntas 1–3 respondidas con argumentos basados en los datos
- [ ] El notebook ejecuta de arriba a abajo sin errores

### Doctorado (adicional)
- [ ] QDA desde cero implementado y verificado contra sklearn
- [ ] Experimento de ablación ejecutado con gráfico
- [ ] Proyección LDA visualizada e interpretada
- [ ] Preguntas 4–6 respondidas con soporte matemático

---

**¿Qué fue lo más difícil de este assignment?** *(escribe aquí)*

**¿Qué queda sin entender o quisieras explorar más?** *(escribe aquí)*